# 04 · Modelos finales, estabilidad y generalización (Etapa III)

**Fases CRISP-DM:** modelado comparativo y evaluación.

1. **Ensambles:** Random Forest (bagging), LightGBM y XGBoost (boosting),
   con búsqueda de hiperparámetros.
2. **Comparación** con la línea base del notebook 03, en clasificación
   binaria y multiclase.
3. **Explicabilidad:** importancia por permutación.
4. **Estabilidad de variables** entre entrenamiento y prueba oficial
   (Concept Drift dentro de IoMT).
5. **Generalización cruzada** entre redes tradicionales (CSE-CIC-IDS2018)
   e IoMT (CICIoMT2024). Esta sección es opcional y se ejecuta solo si
   el CSV de 2018 está descargado (ver `DESCARGA_DATOS.txt`).

In [ ]:
import sys
import time
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ.resolve()))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import loguniform, randint, uniform
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from src.data_loader import (
    CLASES, CARPETA_MODELOS, CARPETA_RESULTADOS,
    cargar_procesado, cargar_variables, cargar_ids2018,
)
from src.features import (
    preprocesador, estabilidad_variables,
    homologar_iomt, homologar_ids2018, VARIABLES_COMUNES,
)
from src.evaluate import evaluar, matriz_confusion, guardar_tabla

config = cargar_variables()
variables = config["variables"]
SEMILLA = config["semilla"]

train = cargar_procesado("train")
val = cargar_procesado("val")
test = cargar_procesado("test")

print("Variables:", len(variables))
print("Entrenamiento:", train.shape, "| Validación:", val.shape, "| Prueba oficial:", test.shape)

## 1. Configuración de la búsqueda de hiperparámetros

La búsqueda se hace sobre una **muestra estratificada** del entrenamiento,
con validación cruzada de 3 particiones y F1 macro como criterio. El mejor
conjunto de hiperparámetros se reentrena luego con todo el entrenamiento.

El desbalance se trata con pesos por muestra (`compute_sample_weight`),
igual para los tres modelos. Aumentar `MUESTRA_BUSQUEDA` o
`ITERACIONES_BUSQUEDA` mejora la búsqueda, pero aumenta el tiempo.

In [ ]:
MUESTRA_BUSQUEDA = 100_000
ITERACIONES_BUSQUEDA = 10


def candidatos():
    return {
        "Random Forest": (
            RandomForestClassifier(n_jobs=-1, random_state=SEMILLA),
            {
                "n_estimators": randint(100, 400),
                "max_depth": [None, 10, 20, 30],
                "min_samples_leaf": randint(1, 20),
                "max_features": ["sqrt", 0.3, 0.5],
            },
        ),
        "LightGBM": (
            LGBMClassifier(n_jobs=-1, random_state=SEMILLA, verbose=-1),
            {
                "n_estimators": randint(100, 600),
                "learning_rate": loguniform(0.02, 0.3),
                "num_leaves": randint(15, 128),
                "min_child_samples": randint(10, 100),
                "subsample": uniform(0.6, 0.4),
                "subsample_freq": [1],
                "colsample_bytree": uniform(0.6, 0.4),
            },
        ),
        "XGBoost": (
            XGBClassifier(tree_method="hist", n_jobs=-1, random_state=SEMILLA),
            {
                "n_estimators": randint(100, 600),
                "learning_rate": loguniform(0.02, 0.3),
                "max_depth": randint(3, 10),
                "min_child_weight": randint(1, 10),
                "subsample": uniform(0.6, 0.4),
                "colsample_bytree": uniform(0.6, 0.4),
            },
        ),
    }


def muestra_estratificada(datos, n, columna="clase"):
    if len(datos) <= n:
        return datos
    return train_test_split(
        datos, train_size=n, stratify=datos[columna], random_state=SEMILLA
    )[0]


def ajustar(nombre, objetivo, variables=variables, datos=train):
    """Busca hiperparámetros y reentrena con todos los datos de `datos`."""
    estimador, espacio = candidatos()[nombre]
    pipeline = Pipeline([("prep", preprocesador()), ("modelo", estimador)])

    estrato = "clase" if "clase" in datos else objetivo
    muestra = muestra_estratificada(datos, MUESTRA_BUSQUEDA, estrato)
    busqueda = RandomizedSearchCV(
        pipeline,
        {f"modelo__{k}": v for k, v in espacio.items()},
        n_iter=ITERACIONES_BUSQUEDA,
        scoring="f1_macro",
        cv=StratifiedKFold(3, shuffle=True, random_state=SEMILLA),
        random_state=SEMILLA,
        refit=False,
    )
    busqueda.fit(
        muestra[variables], muestra[objetivo],
        modelo__sample_weight=compute_sample_weight("balanced", muestra[objetivo]),
    )

    modelo = clone(pipeline).set_params(**busqueda.best_params_)
    inicio = time.perf_counter()
    modelo.fit(
        datos[variables], datos[objetivo],
        modelo__sample_weight=compute_sample_weight("balanced", datos[objetivo]),
    )
    return modelo, {
        "Mejores hiperparámetros": {
            k.removeprefix("modelo__"): v for k, v in busqueda.best_params_.items()
        },
        "F1 macro CV": busqueda.best_score_,
        "Segundos de entrenamiento": time.perf_counter() - inicio,
    }

## 2. Entrenamiento y evaluación

Se entrenan los tres ensambles para la tarea binaria (benigno vs. DDoS)
y para la multiclase (benigno, DDoS SYN y DDoS UDP).

In [ ]:
tareas = {"binaria": "etiqueta", "multiclase": "clase"}
modelos = {}
busquedas = []
resultados = {"binaria": {"val": [], "test": []},
              "multiclase": {"val": [], "test": []}}

for tarea, objetivo in tareas.items():
    for nombre in candidatos():
        print(f"Ajustando {nombre} ({tarea})...")
        modelo, info = ajustar(nombre, objetivo)
        modelos[(tarea, nombre)] = modelo
        busquedas.append({"Tarea": tarea, "Modelo": nombre, **info})

        for particion, datos in [("val", val), ("test", test)]:
            resultados[tarea][particion].append({"Modelo": nombre, **evaluar(
                modelo, datos[variables], datos[objetivo], tarea,
                clase_real=datos["clase"] if tarea == "binaria" else None,
            )})

busquedas = pd.DataFrame(busquedas).set_index(["Tarea", "Modelo"])
display(busquedas)
guardar_tabla(busquedas, "busqueda_hiperparametros")

In [ ]:
tablas = {
    (tarea, particion): pd.DataFrame(filas).set_index("Modelo")
    for tarea, por_particion in resultados.items()
    for particion, filas in por_particion.items()
}

for (tarea, particion), tabla in tablas.items():
    nombre = "validación" if particion == "val" else "prueba oficial"
    print(f"{tarea.capitalize()} — {nombre}")
    display(tabla.round(4))
    guardar_tabla(tabla, f"metricas_{tarea}_ensambles_{particion}")

## 3. Comparación con la línea base

Se agregan los resultados de la regresión logística y del árbol de
decisión obtenidos en el notebook 03 (escenario V2 en el caso binario).

In [ ]:
base_binaria = (
    pd.read_csv(CARPETA_RESULTADOS / "metricas_prueba_oficial.csv",
                index_col=["Escenario", "Modelo"])
    .loc["V2: benigno + SYN + UDP"]
)
base_multi = pd.read_csv(
    CARPETA_RESULTADOS / "metricas_multiclase_base_prueba_oficial.csv",
    index_col="Modelo",
)

comparacion = {
    "binaria": pd.concat([base_binaria, tablas[("binaria", "test")]]),
    "multiclase": pd.concat([base_multi, tablas[("multiclase", "test")]]),
}
columnas_clave = {
    "binaria": ["F1 DDoS", "F1 macro", "Recall SYN", "Recall UDP",
                "Tasa falsos positivos", "Falsas alarmas", "Ataques no detectados"],
    "multiclase": ["F1 macro", "Recall Benigno", "Recall DDoS SYN",
                   "Recall DDoS UDP", "Tasa falsos positivos"],
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (tarea, tabla) in zip(axes, comparacion.items()):
    print(f"{tarea.capitalize()} — prueba oficial")
    display(tabla[columnas_clave[tarea]].round(4))
    tabla["F1 macro"].sort_values().plot.barh(ax=ax)
    ax.set_xlim(max(0, tabla["F1 macro"].min() - 0.05), 1)
    ax.set_title(f"F1 macro en prueba oficial ({tarea})")
plt.tight_layout()
plt.show()

guardar_tabla(comparacion["multiclase"], "comparacion_multiclase_prueba_oficial")
guardar_tabla(comparacion["binaria"], "comparacion_binaria_prueba_oficial")

El mejor modelo se elige según el **F1 macro en validación**, no en la
prueba oficial, para no seleccionar el modelo con los datos de evaluación.

In [ ]:
mejor = {
    tarea: tablas[(tarea, "val")]["F1 macro"].idxmax() for tarea in tareas
}
print("Mejor modelo por tarea (según validación):", mejor)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (tarea, objetivo) in zip(axes, tareas.items()):
    modelo = modelos[(tarea, mejor[tarea])]
    matriz_confusion(
        test[objetivo], modelo.predict(test[variables]), tarea,
        f"{mejor[tarea]} — {tarea}", ax=ax, normalizar="true",
    )
fig.suptitle("Prueba oficial: matriz de confusión del mejor modelo")
plt.tight_layout()
plt.show()

## 4. Explicabilidad: importancia por permutación

Se mide cuánto cae el F1 macro en validación al desordenar cada variable.
A diferencia de la importancia interna de los árboles, este método es
comparable entre modelos distintos.

In [ ]:
modelo_multi = modelos[("multiclase", mejor["multiclase"])]
muestra_val = muestra_estratificada(val, 20_000)

permutacion = permutation_importance(
    modelo_multi, muestra_val[variables], muestra_val["clase"],
    scoring="f1_macro", n_repeats=5, random_state=SEMILLA,
)
importancia = pd.DataFrame({
    "Importancia": permutacion.importances_mean,
    "Desviación": permutacion.importances_std,
}, index=variables).sort_values("Importancia", ascending=False)

display(importancia.head(15).round(4))

importancia.head(15).sort_values("Importancia")["Importancia"].plot.barh(
    figsize=(8, 6), xerr=importancia.head(15).sort_values("Importancia")["Desviación"],
    title=f"Importancia por permutación — {mejor['multiclase']}",
)
plt.xlabel("Caída del F1 macro al permutar la variable")
plt.tight_layout()
plt.show()

## 5. Estabilidad de las variables dentro de IoMT

Para cada clase se compara la distribución de cada variable entre
entrenamiento y prueba oficial:

- **KS** (Kolmogorov-Smirnov): 0 = distribuciones iguales, 1 = sin solapamiento.
- **PSI**: < 0,1 estable; 0,1–0,25 cambio moderado; > 0,25 cambio alto.

Se compara por clase para no confundir el cambio de las variables con
el cambio de proporciones entre clases.

In [ ]:
estabilidad = pd.concat({
    CLASES[c]: estabilidad_variables(
        train[train["clase"] == c], test[test["clase"] == c], variables
    )
    for c in CLASES
}, axis=1)

ks = estabilidad.xs("KS", axis=1, level=1)
resumen_estabilidad = pd.DataFrame({
    "KS medio": ks.mean(axis=1),
    "KS máximo": ks.max(axis=1),
    "PSI medio": estabilidad.xs("PSI", axis=1, level=1).mean(axis=1),
}).join(importancia["Importancia"]).sort_values("KS medio")

display(resumen_estabilidad.round(4))
guardar_tabla(resumen_estabilidad, "estabilidad_variables_iomt")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(resumen_estabilidad["KS medio"], resumen_estabilidad["Importancia"])
for variable, fila in resumen_estabilidad.nlargest(12, "Importancia").iterrows():
    ax.annotate(variable, (fila["KS medio"], fila["Importancia"]),
                fontsize=8, xytext=(4, 2), textcoords="offset points")
ax.axvline(0.3, color="gray", linestyle="--")
ax.set_xlabel("KS medio entre entrenamiento y prueba oficial (mayor = menos estable)")
ax.set_ylabel("Importancia por permutación")
ax.set_title("Importancia frente a estabilidad de las variables")
plt.tight_layout()
plt.show()

Las variables **importantes y estables** (abajo a la izquierda en KS,
arriba en importancia) son candidatas para una detección robusta.
Las variables **importantes e inestables** son un riesgo: el modelo
depende de ellas, pero cambian entre capturas.

### Prueba de robustez

Se reentrena el mejor modelo multiclase, con los mismos hiperparámetros,
sin las variables cuyo KS medio supera `UMBRAL_KS`. Si el desempeño en la
prueba oficial se mantiene o mejora, el modelo no depende de esas variables.

In [ ]:
UMBRAL_KS = 0.3

inestables = resumen_estabilidad.index[resumen_estabilidad["KS medio"] > UMBRAL_KS].tolist()
estables = [v for v in variables if v not in inestables]
print(f"Variables inestables (KS medio > {UMBRAL_KS}):", inestables or "ninguna")

if inestables and estables:
    robusto = clone(modelo_multi).fit(
        train[estables], train["clase"],
        modelo__sample_weight=compute_sample_weight("balanced", train["clase"]),
    )
    robustez = pd.DataFrame([
        {"Variables": f"Todas ({len(variables)})",
         **evaluar(modelo_multi, test[variables], test["clase"], "multiclase")},
        {"Variables": f"Solo estables ({len(estables)})",
         **evaluar(robusto, test[estables], test["clase"], "multiclase")},
    ]).set_index("Variables")
    display(robustez.round(4))
    guardar_tabla(robustez, "robustez_variables_estables")

## 6. Generalización cruzada: redes tradicionales e IoMT

Se usa el archivo de CSE-CIC-IDS2018 del 21-02-2018, que contiene
tráfico benigno y los ataques DDoS-LOIC-UDP y DDoS-HOIC.

**Homologación:** los dos datasets usan extractores distintos
(CICIoMT2024 resume ventanas de paquetes y CIC-IDS2018 resume flujos
con CICFlowMeter). Solo se usan variables con significado equivalente:
proporción de TCP/UDP, proporción de cada bandera TCP, longitudes de
paquete (mínimo, máximo, media, desviación) y tasa de paquetes. Esta
equivalencia es aproximada y se debe reportar como limitación.

Como los ataques no coinciden exactamente entre datasets, la
evaluación cruzada es **binaria** (benigno vs. DDoS).

Experimentos:

| Entrenamiento | Evaluación | Propósito |
|---|---|---|
| IoMT | IoMT | Línea base dentro del dominio |
| Tradicional | Tradicional | Línea base dentro del dominio |
| Tradicional | IoMT | ¿Un NIDS tradicional sirve en IoMT? |
| IoMT | Tradicional | Dirección inversa |

In [ ]:
MUESTRA_DOMINIO = 300_000

ids2018 = cargar_ids2018()
HAY_IDS2018 = ids2018 is not None

if not HAY_IDS2018:
    print("No se encontró CSE-CIC-IDS2018 en data/raw/CSE-CIC-IDS2018/.")
    print("Se omite la generalización cruzada (ver DESCARGA_DATOS.txt).")
else:
    print("Registros CSE-CIC-IDS2018:", len(ids2018))
    display(ids2018["Label"].value_counts().to_frame())

    # IoMT: se parte de los datos limpios con todas las columnas originales,
    # porque la homologación usa variables que el notebook 02 pudo descartar
    iomt_train = homologar_iomt(cargar_procesado("train_limpio"))
    iomt_test = homologar_iomt(cargar_procesado("test_limpio"))

    trad = homologar_ids2018(ids2018)
    trad_train, trad_test = train_test_split(
        trad, test_size=0.30, stratify=trad["Label"], random_state=SEMILLA
    )

    dominios = {
        "IoMT": {
            "train": muestra_estratificada(iomt_train, MUESTRA_DOMINIO, "etiqueta"),
            "test": iomt_test,
        },
        "Tradicional": {
            "train": muestra_estratificada(trad_train, MUESTRA_DOMINIO, "Label"),
            "test": trad_test,
        },
    }
    for dominio, partes in dominios.items():
        print(dominio, {k: len(v) for k, v in partes.items()})

Se usa la familia del mejor modelo binario con hiperparámetros elegidos
por búsqueda en cada dominio de origen. Ningún dato del dominio de
destino se usa para entrenar ni para ajustar hiperparámetros.

In [ ]:
if HAY_IDS2018:
    familia = mejor["binaria"]
    cruzados = {}
    for origen, partes in dominios.items():
        print(f"Ajustando {familia} con el dominio {origen}...")
        cruzados[origen], _ = ajustar(
            familia, "etiqueta", VARIABLES_COMUNES, partes["train"]
        )

    filas = []
    for origen, modelo in cruzados.items():
        for destino, partes in dominios.items():
            datos = partes["test"]
            filas.append({
                "Entrenamiento": origen,
                "Evaluación": destino,
                "Tipo": "Mismo dominio" if origen == destino else "Cruzado",
                **evaluar(modelo, datos[VARIABLES_COMUNES], datos["etiqueta"], "binaria"),
            })
    cruzada = pd.DataFrame(filas).set_index(["Entrenamiento", "Evaluación"])
    display(cruzada.round(4))
    guardar_tabla(cruzada, "generalizacion_cruzada")

    # Degradación: F1 en el dominio de destino entrenando en él, menos F1 cruzado
    degradacion = pd.DataFrame([
        {
            "Origen → destino": f"{origen} → {destino}",
            "F1 macro mismo dominio": cruzada.loc[(destino, destino), "F1 macro"],
            "F1 macro cruzado": cruzada.loc[(origen, destino), "F1 macro"],
        }
        for origen in dominios for destino in dominios if origen != destino
    ]).set_index("Origen → destino")
    degradacion["Degradación"] = (
        degradacion["F1 macro mismo dominio"] - degradacion["F1 macro cruzado"]
    )
    display(degradacion.round(4))
    guardar_tabla(degradacion, "degradacion_cruzada")

In [ ]:
if HAY_IDS2018:
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    for ax, ((origen, destino), _) in zip(axes.ravel(), cruzada.iterrows()):
        datos = dominios[destino]["test"]
        matriz_confusion(
            datos["etiqueta"],
            cruzados[origen].predict(datos[VARIABLES_COMUNES]),
            "binaria", f"{origen} → {destino}", ax=ax, normalizar="true",
        )
    fig.suptitle("Matrices de confusión: entrenamiento → evaluación")
    plt.tight_layout()
    plt.show()

    # Recall por tipo de ataque de 2018 para el modelo entrenado en IoMT
    pred = cruzados["IoMT"].predict(trad_test[VARIABLES_COMUNES])
    print("Modelo IoMT evaluado en tráfico tradicional, por etiqueta original:")
    display(
        pd.Series(pred, index=trad_test.index).groupby(trad_test["Label"]).mean()
        .rename("Proporción predicha como DDoS").to_frame().round(4)
    )

### Estabilidad de las variables comunes entre dominios

Se comparan las distribuciones de cada variable común entre IoMT y la
red tradicional, por separado para benigno y DDoS. Las variables con KS
bajo en ambas clases son las más transferibles entre dominios.

In [ ]:
if HAY_IDS2018:
    iomt = dominios["IoMT"]["train"]
    tradicional = dominios["Tradicional"]["train"]
    entre_dominios = pd.concat({
        nombre: estabilidad_variables(
            iomt[iomt["etiqueta"] == valor],
            tradicional[tradicional["etiqueta"] == valor],
            VARIABLES_COMUNES,
        )
        for valor, nombre in [(0, "Benigno"), (1, "DDoS")]
    }, axis=1)
    entre_dominios[("Resumen", "KS medio")] = (
        entre_dominios.xs("KS", axis=1, level=1).mean(axis=1)
    )
    entre_dominios = entre_dominios.sort_values(("Resumen", "KS medio"))
    display(entre_dominios.round(4))
    guardar_tabla(entre_dominios, "estabilidad_variables_entre_dominios")

    entre_dominios[("Resumen", "KS medio")].plot.barh(
        figsize=(8, 5), title="Cambio de distribución entre IoMT y red tradicional"
    )
    plt.xlabel("KS medio (benigno y DDoS)")
    plt.tight_layout()
    plt.show()

## 7. Guardar los modelos finales

In [ ]:
CARPETA_MODELOS.mkdir(parents=True, exist_ok=True)
for tarea in tareas:
    ruta = CARPETA_MODELOS / f"mejor_{tarea}.joblib"
    joblib.dump({
        "modelo": modelos[(tarea, mejor[tarea])],
        "nombre": mejor[tarea],
        "variables": variables,
        "clases": CLASES,
    }, ruta)
    print(f"{tarea}: {mejor[tarea]} → {ruta}")

## Conclusiones

Completar con los resultados obtenidos, siguiendo los objetivos
específicos del informe:

1. **Desempeño dentro del dominio (objetivos 2, 3 y 5):** mejora de los
   ensambles frente a la línea base en F1 macro, recall por clase y tasa
   de falsos positivos.
2. **Generalización cruzada (objetivo 4):** degradación del F1 al
   entrenar en un dominio y evaluar en otro, y qué ataques dejan de
   detectarse.
3. **Estabilidad de variables (objetivo 6):** variables importantes y
   estables, variables importantes e inestables, y resultado de la
   prueba de robustez.
4. **Limitaciones:** homologación aproximada entre extractores, solo dos
   variantes de DDoS en IoMT y prueba oficial ya examinada.